# 移動平均クロス戦略 - バックテスト分析

このノートブックでは以下を実施します:
1. yfinance でデータ取得
2. テクニカル指標の計算・可視化
3. 移動平均クロス戦略のバックテスト
4. 結果のサマリー表示

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data import MarketDataFetcher, TechnicalIndicators
from src.strategies import MovingAverageCrossStrategy

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True

## 1. データ取得

In [ ]:
fetcher = MarketDataFetcher()
df = fetcher.fetch(symbol='AAPL', start='2022-01-01', end='2024-12-31')
print(f'取得件数: {len(df)} 件')
df.tail()

## 2. テクニカル指標の追加・可視化

In [ ]:
ind = TechnicalIndicators()
df_ind = ind.add_all(df)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 終値 + ボリンジャーバンド + SMA
ax = axes[0]
ax.plot(df_ind.index, df_ind['Close'], label='Close', linewidth=1)
ax.plot(df_ind.index, df_ind['SMA_20'], label='SMA 20', linewidth=1)
ax.plot(df_ind.index, df_ind['SMA_50'], label='SMA 50', linewidth=1)
ax.fill_between(df_ind.index, df_ind['BB_upper'], df_ind['BB_lower'], alpha=0.1, label='Bollinger Bands')
ax.set_title('Price + Indicators')
ax.legend()

# RSI
ax2 = axes[1]
ax2.plot(df_ind.index, df_ind['RSI_14'], color='purple', linewidth=1)
ax2.axhline(70, color='red', linestyle='--', linewidth=0.8, label='Overbought (70)')
ax2.axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
ax2.set_title('RSI (14)')
ax2.set_ylim(0, 100)
ax2.legend()

# 出来高
ax3 = axes[2]
ax3.bar(df_ind.index, df_ind['Volume'], color='steelblue', alpha=0.6)
ax3.set_title('Volume')

plt.tight_layout()
plt.show()

## 3. 移動平均クロス戦略のバックテスト

In [ ]:
strategy = MovingAverageCrossStrategy(short_window=20, long_window=50)
result = strategy.backtest(df)

# シグナルのプロット
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(result.index, result['Close'], label='Close', linewidth=1, alpha=0.7)
ax.plot(result.index, result['SMA_short'], label=f'SMA {strategy.short_window}', linewidth=1)
ax.plot(result.index, result['SMA_long'], label=f'SMA {strategy.long_window}', linewidth=1)

buy  = result[result['signal'] ==  1]
sell = result[result['signal'] == -1]
ax.scatter(buy.index,  buy['Close'],  marker='^', color='green', s=80, label='Buy',  zorder=5)
ax.scatter(sell.index, sell['Close'], marker='v', color='red',   s=80, label='Sell', zorder=5)
ax.set_title('MA Cross Signals')
ax.legend()
plt.show()

In [ ]:
# 累積リターン比較
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(result.index, result['cumulative_returns'],  label='Buy & Hold', linewidth=1.5)
ax.plot(result.index, result['cumulative_strategy'], label='MA Cross Strategy', linewidth=1.5)
ax.set_title('Cumulative Returns')
ax.set_ylabel('Cumulative Return')
ax.legend()
plt.show()

## 4. サマリー

In [ ]:
summary = strategy.summary(df)
print('=== バックテスト結果 ===')
print(f"戦略リターン   : {summary['total_return']*100:.2f}%")
print(f"B&H リターン   : {summary['buy_and_hold_return']*100:.2f}%")
print(f"取引回数       : {summary['num_trades']} 回")
print(f"シャープレシオ : {summary['sharpe_ratio']:.3f}")